# TensorRT-LLM: Compiling Inference Engines Ahead of Time

The full code of [TensorRT-LLM: Compiling Inference Engines Ahead of Time](https://residual-stream.lvtan71.workers.dev/posts/tensorrt-llm-explained/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/tensorrt-llm-explained) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.



**Hardware:** Needs an Apple-silicon Mac: the kernels are written in Metal, which Colab can't run. Run it locally with Jupyter (see the repo's README); the cells that are only arithmetic run anywhere. About 2 min end to end on the post's Apple M4.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

In [ ]:
%pip install -q "torch==2.14.*" "transformers==5.19.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "tensorrt-llm-explained"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `tune.py`

Auto-tuning: times twelve versions of the RMSNorm kernel at four shapes.

**Note:** Apple-silicon Mac only (Metal).

In [ ]:
!{sys.executable} tune.py

The post printed:

```text
microseconds per call; rows: 1, 16, 256, 4096 (fastest marked *)
  32 threads x 2     9.0      8.8     10.3    285.8 
  32 threads x 4     6.6      6.2      8.3    289.3 
  64 threads x 2     6.5      6.4      9.6    299.3 
  64 threads x 4     5.6      5.1      8.3    284.9 
 128 threads x 2     5.5      5.3      9.6    285.1 
 128 threads x 4     4.7      4.3      8.2*   290.0 
 256 threads x 2     4.6      4.8     10.3    300.4 
 256 threads x 4     4.5      4.9      9.3    278.8 
 512 threads x 2     4.3      4.6     12.7    294.5 
 512 threads x 4     4.3      4.2*    11.0    275.1*
1024 threads x 2     4.5      5.5     16.6    288.9 
1024 threads x 4     4.2*     4.8     14.3    277.0 
   1 rows: fastest   4.2 us, slowest 2.1x as long; within 5% of the fastest: 512x2, 512x4, 1024x4
  16 rows: fastest   4.2 us, slowest 2.1x as long; within 5% of the fastest: 128x4, 512x4
 256 rows: fastest   8.2 us, slowest 2.0x as long; within 5% of the fastest: 32x4, 64x4, 128x4
4096 rows: fastest 275.1 us, slowest 1.1x as long; within 5% of the fastest: 32x2, 64x4, 128x2, 256x4, 512x4, 1024x4
```

## `fuse.py`

Fuses Qwen2.5-1.5B-Instruct's RMSNorms and MLP activations; counts operations per decoded token, times decoding and prefill, compares outputs.

**Note:** Apple-silicon Mac only (Metal).

In [ ]:
!{sys.executable} fuse.py

The post printed:

```text
as written: 1425 operations per decoded token;  39.7 ms per decoded token; prefill of 2,048 tokens 2289.3 ms
fused     : 1055 operations per decoded token;  38.0 ms per decoded token; prefill of 2,048 tokens 2121.3 ms
next-token logits over 512 positions: largest difference 0.051 (logits span -20 to 29); same top token at 510 of 512, where the top two differ by under 0.1 at 48
greedy output: the first 23 of 128 tokens are the same
weights read per decoded token: 3.09 GB; at 38.0 ms per token, 81 GB/s
```